# Olivine (I13 scan 198065) – parallel-beam FBP with CIL

This notebook loads the raw Diamond I13 data, sets up a CIL parallel-beam geometry and reconstructs with CIL FBP.
All tomography steps (normalisation, -log, centre of rotation, reconstruction) use CIL. Only metadata reading uses plain `h5py`/`numpy`.

**Data**
- `198065.nxs`: scan metadata (`image_key`, rotation angles `ss1_theta`).
- `hama-198065.hdf`: raw frames at `/entry/data/data`, shape (4081, 1744, 1700), uint16. That's 40 darks, 40 flats and 4001 projections over 0–180°.
  The `.nxs` external links to the frames point to a path that doesn't exist, so the `.hdf` is read directly.

**Assumptions**
- Frame axes are (angle, vertical, horizontal), with the rotation axis vertical.
- The pixel size (`PIXEL_SIZE_UM`) isn't stored in the file. The estimate from the optics is 0.65 µm (6.5 µm pco.edge / 10× objective). The reconstructed values are in 1/µm.

**Things to know about this dataset**
- The sample is a slab wider than the field of view (local/interior tomography). This causes a bright rim at the edge of the reconstruction circle.
- The flat field has concentric structure, which causes ring artefacts. Ring removal (CIL `RingRemover`) is applied to the final data; use the ring-removal sweep to tune it.
- Contrast is very low, so CIL's `image_sharpness` centre-of-rotation method doesn't converge. Instead, this notebook uses `xcorrelation` for the estimate,
  plus a visual sweep of FBP slices so you can check or override it (`COR_OFFSET_PX`).

In [ ]:
import os, time
import h5py
import numpy as np
import matplotlib.pyplot as plt

from cil.io.utilities import HDF5_utilities
from cil.framework import AcquisitionGeometry
from cil.processors import Normaliser, TransmissionAbsorptionConverter, CentreOfRotationCorrector, Binner, RingRemover
from cil.recon import FBP
from cil.utilities.display import show2D, show_geometry


def fov_mask(shape, frac=0.45):
    # disc covering the inner part of the reconstruction circle (excludes the bright truncation rim)
    ny, nx = shape[-2:]
    yy, xx = np.mgrid[:ny, :nx]
    return (yy - ny / 2) ** 2 + (xx - nx / 2) ** 2 < (frac * min(nx, ny)) ** 2


def fov_range(arr, q=(1, 99)):
    # display range from percentiles inside the field-of-view disc
    return tuple(np.percentile(arr[..., fov_mask(arr.shape)], q))

## Configuration
- `MODE='slices'` reconstructs the detector rows selected by `SLICE_ROWS`. The rows must be evenly spaced (a `slice` with a step), so they form a valid
  3D geometry with vertical pixel size `step × PIXEL_SIZE_UM`. Each selected row takes about 40 s to read from the share.
- `MODE='full'` reconstructs the whole volume. This reads 23 GB and needs about 47 GB of RAM for the float32 projections, plus about 20 GB for the volume at `FULL_BIN=1`.
  `FULL_BIN` bins the vertical and horizontal detector axes after normalisation.

In [ ]:
DATA_DIR = '/mnt/share/olivine_data/olivine data'
NXS = os.path.join(DATA_DIR, '198065.nxs')
HDF = os.path.join(DATA_DIR, 'hama-198065.hdf')
DSET = '/entry/data/data'

MODE = 'slices'                       # 'slices' or 'full'
SLICE_ROWS = slice(436, 1309, 436)    # detector rows 436, 872, 1308 (used in 'slices' mode)
FULL_BIN = 1                          # detector binning in 'full' mode

PIXEL_SIZE_UM = 1                  # detector pixel size at the sample (µm) - TODO:
COR_OFFSET_PX = None                  # final COR offset (detector pixels); None: use CIL xcorrelation estimate
COR_COARSE_PX = np.arange(-40, 41, 5) # offsets for the coarse sweep (middle row)
COR_FINE_CENTRE_PX = None             # centre of the fine sweep; None: xcorrelation estimate
COR_FINE_HALFWIDTH_PX = 3             # fine sweep covers centre ± this
COR_FINE_STEP_PX = 0.5
COR_ZOOM = (slice(700, 1300), slice(550, 1150))   # (y, x) region shown in the sweeps, unbinned pixels (centre + grain in row 436)
COR_SWEEP_RING_REMOVAL = True         # RingRemover (RING_* settings) on copies of the sinograms used for the COR sweeps

RING_REMOVAL = True                   # apply CIL RingRemover to the data before the final FBP
RING_DECNUM = 6                       # wavelet decomposition levels (more = removes broader rings)
RING_WNAME = 'db10'
RING_SIGMA = 6                        # damping of the stripe filter (larger = stronger)
RING_SWEEP = [(4, 1.5), (4, 3), (6, 3), (6, 6), (8, 6)]   # (decNum, sigma) settings compared in the sweep
BACKEND = 'tigre'
FRAME_CHUNK = 200                     # frames per HDF5 read

## Metadata

In [ ]:
with h5py.File(NXS, 'r') as f:
    image_key = f['entry1/hama_sw_hdf/image_key'][()].astype(int)
    theta = f['entry1/hama_sw_hdf/ss1_theta'][()]

meta = HDF5_utilities.get_dataset_metadata(HDF, DSET)
print(meta)
n_frames, n_vert, n_horiz = meta['shape']
assert n_frames == len(image_key)

def frame_range(key):
    idx = np.where(image_key == key)[0]
    assert np.all(np.diff(idx) == 1), 'frames of this type are not contiguous'
    return slice(int(idx[0]), int(idx[-1]) + 1)

dark_sel, flat_sel, proj_sel = frame_range(2), frame_range(1), frame_range(0)
theta_proj = theta[proj_sel]
n_proj = len(theta_proj)
print(f'darks {dark_sel}, flats {flat_sel}, projections {proj_sel}')
print(f'{n_proj} projections, {theta_proj.min():.3f} to {theta_proj.max():.3f} deg, mean step {np.diff(theta_proj).mean():.4f} deg')

## Look at the raw data

In [ ]:
row_sel = SLICE_ROWS if MODE == 'slices' else slice(None)
rows = range(n_vert)[row_sel]
row_step = row_sel.step or 1
assert len(rows) >= 2, 'select at least two rows'
print(f'{MODE} mode: {len(rows)} detector rows', list(rows) if MODE == 'slices' else '')

frame = lambda i: HDF5_utilities.read(HDF, DSET, source_sel=(slice(i, i + 1), slice(None), slice(None)))[0]
frames = [frame(dark_sel.start), frame(flat_sel.start), frame(proj_sel.start), frame(proj_sel.start + n_proj // 2)]
show2D(frames, title=['dark', 'flat', 'projection 0 deg', 'projection 90 deg'],
       fix_range=[tuple(np.percentile(f, [0.5, 99.5])) for f in frames],
       origin='upper-left', num_cols=4, size=(20, 5));

## Geometry
Parallel-beam 3D geometry in µm. In slice mode, the vertical pixel size is `row_step × PIXEL_SIZE_UM`.

In [ ]:
ag = (AcquisitionGeometry.create_Parallel3D(units='um')
      .set_panel((n_horiz, len(rows)), pixel_size=(PIXEL_SIZE_UM, PIXEL_SIZE_UM * row_step))
      .set_angles(theta_proj, angle_unit='degree')
      .set_labels(['angle', 'vertical', 'horizontal']))
print(ag)
show_geometry(ag);

## Load projections, flats and darks

In [ ]:
t0 = time.time()
data = ag.allocate(None)
arr = data.as_array()
for start in range(0, n_proj, FRAME_CHUNK):
    stop = min(start + FRAME_CHUNK, n_proj)
    HDF5_utilities.read_to(HDF, DSET, arr,
                           source_sel=(slice(proj_sel.start + start, proj_sel.start + stop), row_sel, slice(None)),
                           dest_sel=(slice(start, stop), slice(None), slice(None)))
    print(f'\rread {stop}/{n_proj} projections ({time.time() - t0:.0f} s)', end='')

flat = HDF5_utilities.read(HDF, DSET, source_sel=(flat_sel, row_sel, slice(None))).mean(axis=0)
dark = HDF5_utilities.read(HDF, DSET, source_sel=(dark_sel, row_sel, slice(None))).mean(axis=0)
print(f'\nloaded {data.shape} in {time.time() - t0:.0f} s')

## Preprocessing: flat/dark normalisation and -log
First, before/after views of the CIL `Normaliser` (flat/dark correction) in projection space and in sinogram space. Then the -log conversion.

### Projection space
Full frames at 0° and 90° are read separately, because in slice mode `data` only holds a few detector rows.
They're normalised with the full-frame mean flat and dark.

In [ ]:
flat_full = HDF5_utilities.read(HDF, DSET, source_sel=(flat_sel, slice(None), slice(None))).mean(axis=0)
dark_full = HDF5_utilities.read(HDF, DSET, source_sel=(dark_sel, slice(None), slice(None))).mean(axis=0)

idx = [0, n_proj // 2]
ag_proj = (AcquisitionGeometry.create_Parallel3D(units='um')
           .set_panel((n_horiz, n_vert), pixel_size=PIXEL_SIZE_UM)
           .set_angles(theta_proj[idx], angle_unit='degree'))
proj_raw = ag_proj.allocate(None)
for k, i in enumerate(idx):
    HDF5_utilities.read_to(HDF, DSET, proj_raw.as_array(),
                           source_sel=(slice(proj_sel.start + i, proj_sel.start + i + 1), slice(None), slice(None)),
                           dest_sel=(slice(k, k + 1), slice(None), slice(None)))
proj_norm = Normaliser(flat_field=flat_full, dark_field=dark_full)(proj_raw)

prange = lambda a: tuple(np.percentile(a, [0.5, 99.5]))
panels = [proj_raw.get_slice(angle=0), proj_raw.get_slice(angle=1), proj_norm.get_slice(angle=0), proj_norm.get_slice(angle=1)]
show2D(panels,
       title=[f'before: raw {theta_proj[i]:.0f} deg' for i in idx] + [f'after: flat/dark corrected {theta_proj[i]:.0f} deg' for i in idx],
       fix_range=[prange(p.as_array()) for p in panels], origin='upper-left', num_cols=2, size=(16, 15));
del flat_full, dark_full, proj_raw, proj_norm, panels

### Sinogram space
The middle selected row, before and after the in-place `Normaliser` on `data`.

In [ ]:
v_mid = data.get_dimension_size('vertical') // 2
sino_raw = data.get_slice(vertical=v_mid).copy()
Normaliser(flat_field=flat, dark_field=dark)(data, out=data)
sino_norm = data.get_slice(vertical=v_mid).copy()

show2D([sino_raw, sino_norm], title=['before: raw counts', 'after: flat/dark corrected (transmission)'],
       fix_range=[prange(s.as_array()) for s in (sino_raw, sino_norm)], num_cols=2, size=(16, 8));
del sino_raw, sino_norm

### -log (absorption)

In [ ]:
TransmissionAbsorptionConverter()(data, out=data)

if MODE == 'full' and FULL_BIN > 1:
    data = Binner(roi={'vertical': (None, None, FULL_BIN), 'horizontal': (None, None, FULL_BIN)})(data)

mid = data.get_dimension_size('vertical') // 2
if MODE == 'slices':
    show2D(data, slice_list=('vertical', mid), title='absorption sinogram (middle row)', size=(8, 8));
else:
    show2D(data, slice_list=[('vertical', mid), ('angle', 0)],
           title=['absorption sinogram (middle row)', 'absorption projection 0 deg'], num_cols=2, size=(16, 6));

## Centre of rotation
Automatic methods are unreliable on this dataset. `image_sharpness` doesn't converge because of the low contrast.
`xcorrelation` is biased towards 0 because the flat-field ring pattern is roughly symmetric about the detector centre. So the offset is chosen by eye:
1. **Coarse sweep** on the middle row over `COR_COARSE_PX`.
2. **Fine sweep** in steps of `COR_FINE_STEP_PX` around `COR_FINE_CENTRE_PX`, for each selected row.
3. Set `COR_OFFSET_PX` to the best value and re-run from here.

**What to look for:** at the wrong offset, real features (grain boundaries, small inclusions) smear into arcs or double up.
Note that a defective detector column always reconstructs as a sharp *half* ring in a 180° scan, whatever the offset; only its radius changes.
By default the sweeps run on ring-removed **copies** of the sinograms (`COR_SWEEP_RING_REMOVAL`) so that doesn't distract. The final reconstruction is not ring-removed.
Change `COR_ZOOM` to look at other regions, e.g. `(slice(900, 1300), slice(550, 1100))` (features in row 436)
or `(slice(1400, 1600), slice(300, 1400))` (slab surface).

In [ ]:
sino = data.get_slice(vertical=mid)
cor_est = CentreOfRotationCorrector.xcorrelation()(sino).geometry.get_centre_of_rotation(distance_units='pixels')['offset'][0]
print(f'xcorrelation estimate: {cor_est:.2f} px')

b = 1 if MODE == 'slices' else FULL_BIN
zoom = tuple(slice(z.start // b, z.stop // b) for z in COR_ZOOM)


def sweep_sinogram(v):
    s = data.get_slice(vertical=v)
    return RingRemover(RING_DECNUM, RING_WNAME, RING_SIGMA, info=False)(s) if COR_SWEEP_RING_REMOVAL else s


def fbp_at(s, offset):
    s = s.copy()
    s.geometry.set_centre_of_rotation(float(offset), distance_units='pixels')
    return FBP(s, s.geometry.get_ImageGeometry(), backend=BACKEND).run(verbose=0).as_array()

### Coarse sweep (middle row)

In [ ]:
s = sweep_sinogram(mid)
recs = [fbp_at(s, o)[zoom] for o in COR_COARSE_PX]
show2D(recs, title=[f'offset {o:g} px' for o in COR_COARSE_PX], fix_range=fov_range(recs[len(recs) // 2]),
       origin='upper-left', num_cols=4, size=(20, 5 * -(-len(recs) // 4)));

### Fine sweep (each selected row)
Set `COR_FINE_CENTRE_PX` to the best coarse offset. If the best offset differs between the top and bottom rows, the rotation axis is tilted
relative to the detector. That can be corrected with `data.geometry.set_centre_of_rotation(offset, angle=..., angle_units='degree')`.

In [ ]:
centre = cor_est if COR_FINE_CENTRE_PX is None else COR_FINE_CENTRE_PX
fine = centre + np.arange(-COR_FINE_HALFWIDTH_PX, COR_FINE_HALFWIDTH_PX + 1e-6, COR_FINE_STEP_PX)
n_v = data.get_dimension_size('vertical')
sweep_rows = range(n_v) if MODE == 'slices' else [n_v // 4, n_v // 2, 3 * n_v // 4]

for v in sweep_rows:
    det_row = rows[v] if MODE == 'slices' else v * b
    s = sweep_sinogram(v)
    recs = [fbp_at(s, o)[zoom] for o in fine]
    show2D(recs, title=[f'row {det_row}, offset {o:.1f} px' for o in fine], fix_range=fov_range(recs[len(recs) // 2]),
           origin='upper-left', num_cols=5, size=(20, 4 * -(-len(recs) // 5)));

In [ ]:
cor = cor_est if COR_OFFSET_PX is None else COR_OFFSET_PX
data.geometry.set_centre_of_rotation(cor, distance_units='pixels')
print(f'using centre-of-rotation offset {cor:.2f} px')
print(data.geometry)

## Ring removal
CIL `RingRemover` is a wavelet-FFT stripe filter applied to each sinogram. Vertical stripes in the sinogram (fixed detector errors) become rings in the slice.
- `decNum` is the number of wavelet levels. More levels remove broader stripes, which matters for the broad rings from the flat-field mismatch.
- `sigma` sets the filter strength.

Stronger settings remove more rings but can smear real low-contrast features, so check the grain boundaries and not just the rings.
The sweep below uses the middle row at the applied centre of rotation. Set `RING_DECNUM`/`RING_SIGMA` (or `RING_REMOVAL = False`) and re-run from here.

In [ ]:
s_mid = data.get_slice(vertical=mid)
settings = [None] + RING_SWEEP
recs = []
for st in settings:
    s = s_mid if st is None else RingRemover(st[0], RING_WNAME, st[1], info=False)(s_mid)
    recs.append(FBP(s, s.geometry.get_ImageGeometry(), backend=BACKEND).run(verbose=0).as_array())
titles = ['no ring removal' if st is None else f'decNum={st[0]}, sigma={st[1]}' for st in settings]
rng = fov_range(recs[0])

show2D([r[zoom] for r in recs], title=titles, fix_range=rng, origin='upper-left', num_cols=3, size=(21, 7 * -(-len(recs) // 3)));
show2D(recs, title=titles, fix_range=rng, origin='upper-left', num_cols=3, size=(21, 7 * -(-len(recs) // 3)));

In [ ]:
# uncorrected middle row, kept for the before/after comparison
rec_before = recs[0]

if RING_REMOVAL:
    t0 = time.time()
    RingRemover(RING_DECNUM, RING_WNAME, RING_SIGMA, info=False)(data, out=data)   # ~0.4 s per slice
    print(f'ring removal (decNum={RING_DECNUM}, sigma={RING_SIGMA}) took {time.time() - t0:.0f} s')

## FBP reconstruction

In [ ]:
t0 = time.time()
ig = data.geometry.get_ImageGeometry()
fbp = FBP(data, ig, backend=BACKEND)
if MODE == 'full':
    fbp.set_split_processing(slices_per_chunk=32)
recon = fbp.run(verbose=0)
print(f'reconstructed {recon.shape} in {time.time() - t0:.0f} s')

## Results (attenuation in 1/µm)

In [ ]:
n_z = recon.get_dimension_size('vertical')
lo, hi = fov_range(recon.as_array())

after_title = f'ring removal decNum={RING_DECNUM}, sigma={RING_SIGMA}' if RING_REMOVAL else 'no ring removal'
show2D([rec_before, recon.get_slice(vertical=mid).as_array()], title=['middle row: no ring removal', f'middle row: {after_title}'],
       fix_range=(lo, hi), origin='upper-left', num_cols=2, size=(16, 8));
if MODE == 'slices':
    show2D([recon] * n_z, slice_list=[('vertical', i) for i in range(n_z)],
           title=[f'detector row {r}' for r in rows], fix_range=(lo, hi),
           origin='upper-left', num_cols=n_z, size=(7 * n_z, 7));
else:
    show2D(recon, slice_list=[('vertical', n_z // 2), ('horizontal_y', recon.shape[1] // 2), ('horizontal_x', recon.shape[2] // 2)],
           fix_range=(lo, hi), num_cols=3, size=(21, 7));
    show2D(recon, slice_list=[('vertical', k) for k in np.linspace(0, n_z - 1, 6, dtype=int)[1:-1]],
           fix_range=(lo, hi), origin='upper-left', num_cols=4, size=(24, 6));

In [ ]:
# histogram inside the field-of-view disc (avoids the truncation rim)
vals = recon.as_array()[..., fov_mask(recon.shape)].ravel()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(vals, bins=300, range=tuple(np.percentile(vals, [0.1, 99.9])), color='0.3')
ax.set_xlabel('attenuation (1/µm)')
ax.set_ylabel('voxel count')
ax.set_title('Reconstructed values inside the field of view')
plt.show()

## Optional: save the reconstruction

In [ ]:
# from cil.io import TIFFWriter
# TIFFWriter(data=recon, file_name='/path/to/output/olivine_fbp/slice').write()